In [1]:
import os
os.environ['HF_HOME'] = '/media/my_drives/DATA4/models'
os.environ['TRANSFORMERS_CACHE'] = '/media/my_drives/DATA4/models'
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.chdir('/media/my_drives')
import pandas as pd
from tqdm import tqdm
import glob
import itertools
import random
import io
import shutil
import numpy as np
import re
from PIL import Image
from sklearn.metrics import accuracy_score
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from datetime import date
import time
random.seed(1)

/media/my_drives/DATA1/environments/maximilian/CitationPrediction/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/my_drives/DATA1/environments/maximilian/CitationPrediction/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
def resize_img(image, max_dim:int):
    width, height = image.size
    if width > height:
        new_width = max_dim
        new_height = int((new_width / width) * height)
    else:
        new_height = max_dim
        new_width = int((new_height / height) * width)
    return image.resize((new_width, new_height))

def is_openable(path):
    try:
        with Image.open(path):
            return True
    except IOError:
        return False

def add_drive_path(path):
    if path.startswith('/media/my_drives/DATA4/data/image_benchmark_phi') == False:
        return f'/media/my_drives/DATA4/data/image_benchmark_phi/{path}'
    else:
        return path

def is_actual_prediction(prediction: str):
    if any(word in prediction.lower() for word in ['sorry', 'cant']):
        return 'No prediction available'
    return prediction

In [ ]:
from openai import OpenAI
import base64

os.environ["OPENAI_API_KEY"] = "OPEN_API_KEY"
client = OpenAI()

In [4]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def get_json_class_schema(classes:list):
    def get_list(classes:list):
        output = {}
        for pred_class in classes:
            output[pred_class] = {
            "type": "number",
            "minimum": 0,
            "maximum": 1
          }
        return output
    class_schema = {
    "format": {
      "type": "json_schema",
      "name": "probabilities",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": get_list(classes),
        "required": classes,
        "additionalProperties": False
      }}}
    return class_schema

def predict_gpt(img_path, prompt, classes):
    base64_image = encode_image(img_path)
    class_schema = get_json_class_schema(classes)

    response = client.responses.create(
        model="gpt-5-2025-08-07",
        input=[{
            "role": "system",
            "content": "You are an image classification model. Always return valid JSON with class probabilities that sum to 1."
        },
               {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }],
        text=class_schema
    )
    
    return response.output_text

In [ ]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

PRED_COL = 'prediction_gpt5_allclassprobs'

In [ ]:
for DATASET in tqdm(datasets):
    df = pd.read_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Training_1000/{DATASET}.csv')
    classes = list(df.label.unique())
    
    if DATASET == 'KonIQ':
        prompt = f'Classify which aesthetic score measured as a 5-point likert scale is best fitting for the image. Return probabilities for all classes in JSON.'
    elif DATASET == 'GenImgNet':
        prompt = f'Classify whether the image is in the top 33 percent of aesthetic images, mid 33 percent or lowest 33 percent of aesthetic images. Return probabilities for all classes in JSON.'
    elif DATASET == 'AIDA': 
        prompt = f'Classify whether the image is in the top 33 percent of well selling car advertising images, mid 33 percent or lowest 33 percent of well selling car advertising images. Return probabilities for all classes in JSON.'
    else:
        prompt = f'Classify the best fitting class to describe the image by choosing one of the following classes. Return probabilities for all classes in JSON.'

    if PRED_COL not in df.columns:
        df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        response = predict_gpt(df.at[i, 'image_path'], prompt, classes)
        df.at[i, PRED_COL] = response

        if i % 10 == 0:
            df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Training_1000/{DATASET}.csv', index=False)
    df.to_csv(f'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Training_1000/{DATASET}.csv', index=False)

  0%|                                                     | 0/1 [00:00<?, ?it/s]